### Import modules and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [ ]:
water_path = PATHS['raw_data_notebooks'] / 'water.csv'
water_pd = pd.read_csv(water_path)
water_pd.head()

In [ ]:
water_pd.info()

### Renaming columns

In [ ]:
columns_dict = {'DATE': 'date', 'CURRENT_WATER': 'storage', 'ID': 'id'}
water_pd.rename(columns=columns_dict, inplace=True)

In [ ]:
water_pd.head()

### Date column to datetime

In [ ]:
# The exact time is irrelevant
water_pd_split = water_pd['date'].str.rsplit(pat='/', n=1, expand=True)
print(f"We split the dataframe conveniently: \n{water_pd_split}")
year_counts = water_pd_split[1].str.split(' ').str[0].value_counts().sort_index()

#### Visualize the distribution of years

In [ ]:
# Seaborn barplot to visualize the count of different years
plt.figure(figsize=(10, 6))
sns.barplot(x=year_counts.index, y=year_counts.values, hue=year_counts.index, palette='viridis', legend=False)
plt.title('Count of Different Years in Water Data')
plt.xlabel('Year')
plt.ylabel('Count')
plt.tight_layout()

### The 85 threshold has been decided in the EDA file. To ease the comprehension of this notebook, the plot has been done here too

In [ ]:
two_digit_years = water_pd_split[1].str.split(' ').str[0].astype(int)
print(f"Two digit years: \n{two_digit_years}")

complete_years = two_digit_years.apply(lambda x: x + 1900 if x >= 85 else x + 2000)
print(f"Full years: \n{complete_years}")

In [ ]:
definite_date_column = water_pd_split[0] + '/' + complete_years.astype(str)
print(f"Definite date column: \n {definite_date_column}")

water_pd['date'] = pd.to_datetime(definite_date_column, format='%d/%m/%Y')

### Save current cleaning and developing water.ipynb at EDA folder

In [ ]:
cleaned_water_path = PATHS['pre_EDA'] / 'water_for_EDA.csv'
cleaned_water_path.parent.mkdir(parents=True, exist_ok=True)
water_pd.to_csv(cleaned_water_path, index=False)

### Handling Missing Values as concluded at EDA

In [ ]:
water_pd.isna().sum()

In [ ]:
# As there are only two rows with missing storage, we can drop them (maybe interpolate them later)
water_pd = water_pd.dropna()
water_pd.isna().sum()

### Setting index every week, and treating NaNs as stated at EDA

In [ ]:
filter_mask = ((water_pd['id'] == 396) & (water_pd['date'].dt.year < 2005)) | ((water_pd['id'] == 376) & (water_pd['date'].dt.year < 2012))
water_pd_filtered = water_pd[filter_mask == False]
water_pd_filtered

In [ ]:
def reindex_weekly(group): # This function is applied to every reservoir
    group_id = group.name
    # Create a complete weekly date range for this id
    full_range = pd.date_range(start=group['date'].min(), end=group['date'].max(), freq='7D')
    group = group.set_index('date').reindex(full_range)
    group['id'] = group_id
    group = group.reset_index().rename(columns={'index': 'date'})
    return group

In [ ]:
# Drop id before groupby to avoid a future warning (even though the one we apply groupby to has the id column)
water_pd_filtered_full = water_pd_filtered.drop(columns=['id']).groupby(water_pd_filtered['id'], group_keys=False).apply(reindex_weekly)

### Add an imputed feature to be able to know in the future if the storage was filled

In [ ]:
water_pd_filtered_full['storage_imputed'] = water_pd_filtered_full['storage'].isna().astype(int)

### Fill backward the imputed values

In [ ]:
water_pd_filtered_full.loc[:, 'storage'] = water_pd_filtered_full['storage'].fillna(method='bfill')
water_pd_filtered_full

### Converting storage to Integer

In [ ]:
water_pd = water_pd_filtered_full
# Let's ensure that every float is actually an int
print(f"Number of rows: {len(water_pd)}")
print(f"Number of int at storage: {len(water_pd[water_pd['storage'] % 1 == 0])}")

In [ ]:
water_pd['storage'] = water_pd['storage'].astype(int)

### Saving cleaned data

In [ ]:
cleaned_water_path = PATHS['cleaned_data_notebooks'] / 'water_cleaned.parquet'
cleaned_water_path.parent.mkdir(parents=True, exist_ok=True)
water_pd.to_parquet(cleaned_water_path, index=False)